In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.tree import DecisionTreeClassifier
import warnings, os

warnings.filterwarnings("ignore")
os.environ["OMP_NUM_THREADS"] = "1"

df = pd.read_csv("../data/sessions.csv")

ENDING_COLORS = {
    "Balanced Developer": "#534AB7",
    "Burnout Genius":     "#E24B4A",
    "The Grinder":        "#D85A30",
    "The Quiet Life":     "#1D9E75",
    "The Regret":         "#888780",
}

print(f"Rows: {len(df)}  |  Runs: {df['player_id'].nunique()}")
print(df["ending"].value_counts())

Rows: 154  |  Runs: 11
ending
Balanced Developer    42
The Grinder           42
Burnout Genius        28
The Quiet Life        28
The Regret            14
Name: count, dtype: int64


In [2]:
CHOICES = ["Study / Learn", "Work harder", "Rest / Invest in self", "Take a risk", "Invest socially"]

ENDING_CONDITIONS = [
    ("Balanced Developer",     lambda s, sc, rw: all(s[k] >= 55 for k in ["knowledge","money","health","happiness"])),
    ("Burnout Genius",         lambda s, sc, rw: s["knowledge"] >= 85 and s["health"] <= 30),
    ("The Grinder",            lambda s, sc, rw: s["money"] >= 70 and s["happiness"] <= 35),
    ("The Quiet Life",         lambda s, sc, rw: s["health"] + s["happiness"] >= 150 and s["money"] <= 40),
    ("Perpetual Student",      lambda s, sc, rw: s["knowledge"] >= 80 and s["money"] <= 35),
    ("Accidental Millionaire", lambda s, sc, rw: s["money"] >= 85 and rw >= 3),
    ("Social Butterfly",       lambda s, sc, rw: sc >= 4 and s["happiness"] >= 60),
    ("The Regret",             lambda s, sc, rw: any(s[k] <= 20 for k in ["knowledge","money","health","happiness"])),
]

def apply_choice(stats, choice, consec):
    s = stats.copy()
    risk_won = False
    if choice == "Study / Learn":
        s["knowledge"] = min(100, s["knowledge"] + np.random.randint(8, 18))
        s["happiness"] = max(0,   s["happiness"] - np.random.randint(3, 8))
        s["health"]    = max(0,   s["health"]    - np.random.randint(3, 8))
    elif choice == "Work harder":
        s["money"]     = min(100, s["money"]     + np.random.randint(8, 18))
        s["health"]    = max(0,   s["health"]    - np.random.randint(4, 10))
        s["happiness"] = max(0,   s["happiness"] - np.random.randint(4, 10))
    elif choice == "Rest / Invest in self":
        s["health"]    = min(100, s["health"]    + np.random.randint(8, 18))
        s["happiness"] = min(100, s["happiness"] + np.random.randint(5, 12))
    elif choice == "Invest socially":
        s["happiness"] = min(100, s["happiness"] + np.random.randint(8, 18))
        s["money"]     = max(0,   s["money"]     - np.random.randint(3, 8))
    elif choice == "Take a risk":
        luck = s["happiness"] / 100
        if np.random.random() < luck * 0.6 + 0.2:
            s["money"]     = min(100, s["money"]     + np.random.randint(10, 25))
            s["happiness"] = min(100, s["happiness"] + np.random.randint(5, 15))
            risk_won = True
        else:
            s["health"]    = max(0, s["health"] - np.random.randint(10, 20))
            s["money"]     = max(0, s["money"]  - np.random.randint(5, 15))
    # burnout penalty
    if consec >= 3 and choice in ["Study / Learn", "Work harder"]:
        s["health"]    = max(0, s["health"]    - 15)
        s["happiness"] = max(0, s["happiness"] - 10)
    # momentum bonus
    if consec == 2:
        bonus = {"Study / Learn": "knowledge", "Work harder": "money",
                 "Rest / Invest in self": "health", "Invest socially": "happiness"}.get(choice)
        if bonus:
            s[bonus] = min(100, s[bonus] + 5)
    return s, risk_won

def get_ending(stats, social_count, risk_wins):
    for name, cond in ENDING_CONDITIONS:
        if cond(stats, social_count, risk_wins):
            return name
    return "Average Life"

def simulate_game(weights=None):
    stats = {"knowledge": 30, "money": 20, "health": 70, "happiness": 60}
    prev, consec, social_count, risk_wins = None, 0, 0, 0
    trajectory = []
    for _ in range(14):
        trajectory.append(stats.copy())
        choice = np.random.choice(CHOICES, p=weights)
        consec = consec + 1 if choice == prev else 1
        prev = choice
        if choice == "Invest socially":
            social_count += 1
        stats, won = apply_choice(stats, choice, consec)
        if won:
            risk_wins += 1
    return get_ending(stats, social_count, risk_wins), trajectory

print("Engine ready.")

Engine ready.


In [3]:
np.random.seed(42)
N = 10_000

STRATEGIES = {
    "Random":       [0.20, 0.20, 0.20, 0.20, 0.20],
    "Pure Grinder": [0.05, 0.70, 0.05, 0.10, 0.10],
    "Pure Scholar": [0.60, 0.10, 0.10, 0.10, 0.10],
    "Wellness":     [0.05, 0.05, 0.50, 0.10, 0.30],
    "Gambler":      [0.05, 0.10, 0.10, 0.65, 0.10],
    "Social":       [0.05, 0.10, 0.10, 0.05, 0.70],
    "Balanced":     [0.20, 0.20, 0.20, 0.20, 0.20],
}

mc_results = {}
for strat, weights in STRATEGIES.items():
    endings = [simulate_game(weights)[0] for _ in range(N)]
    mc_results[strat] = pd.Series(endings).value_counts(normalize=True).mul(100).round(1)
    print(f"{strat:16s}  ->  {mc_results[strat].index[0]} ({mc_results[strat].iloc[0]:.0f}%)")

Random            ->  Average Life (26%)
Pure Grinder      ->  The Grinder (86%)
Pure Scholar      ->  Burnout Genius (82%)
Wellness          ->  The Quiet Life (78%)
Gambler           ->  Accidental Millionaire (71%)
Social            ->  The Quiet Life (83%)
Balanced          ->  Average Life (26%)


In [ ]:
all_endings = sorted(set(e for s in mc_results.values() for e in s.index))
heat = pd.DataFrame({s: mc_results[s].reindex(all_endings, fill_value=0) for s in STRATEGIES}).T

fig, ax = plt.subplots(figsize=(13, 5))
im = ax.imshow(heat.values, aspect="auto", cmap="YlOrRd", vmin=0, vmax=100)
ax.set_xticks(range(len(all_endings)))
ax.set_xticklabels(all_endings, rotation=30, ha="right", fontsize=9)
ax.set_yticks(range(len(heat)))
ax.set_yticklabels(heat.index)
for i in range(len(heat)):
    for j in range(len(all_endings)):
        val = heat.values[i, j]
        if val > 0:
            ax.text(j, i, f"{val:.0f}%", ha="center", va="center",
                    fontsize=8, color="white" if val > 55 else "black")
plt.colorbar(im, ax=ax, label="% of 10,000 games")
ax.set_title("Monte Carlo: ending probability by strategy")
plt.tight_layout()
plt.show()